In [55]:
#랜덤시드 고정
import pandas as pd
import random
import os
import numpy as np

from sklearn.ensemble import VotingRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
import catboost
from catboost import CatBoostRegressor


In [41]:
#랜덤시드 고정
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42) # Seed 고정

In [42]:
#랜덤시드 고정
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42) # Seed 고정

In [43]:
#데이터 불러오기 
train= pd.read_csv('  자동차가격예측대회/train.csv')
test = pd.read_csv('  자동차가격예측대회/test.csv')

In [44]:
train.head()

,ID,생산년도,모델출시년도,브랜드,차량모델명,판매도시,판매구역,주행거리,배기량,압축천연가스(CNG),경유,가솔린,하이브리드,액화석유가스(LPG),가격
0,TRAIN_00000,2018,2014,skoda,fabia,KAT,SLA,85231,999,0,0,1,0,0,51.74
1,TRAIN_00001,2010,2006,toyota,auris,RKO,SWI,135000,1598,0,0,1,0,0,41.47
2,TRAIN_00002,2002,2002,mercedes-benz,clk-klasa,GNI,WIE,255223,1796,0,0,1,0,0,17.81
3,TRAIN_00003,2006,2001,nissan,x-trail,EHX,WIE,238000,2184,0,1,0,0,0,18.20
4,TRAIN_00004,2007,2007,fiat,bravo,OSW,MAL,251000,1910,0,1,0,0,0,17.55


In [45]:
def modelNmake(data):
    data = data[(data['모델출시년도'] - data['생산년도']) < 12]
    
    data.loc[(data['모델출시년도'] - data['생산년도']) > 2, '생산년도'] = data['모델출시년도'] + 2
    
    return data


In [46]:
#이상치처리 

# 생산년도 1977~1988 사이 데이터(6개) 드롭
train = train[~((train['생산년도'] >= 1977) & (train['생산년도'] <= 1988))]

#모델출시년도, 생산년도 처리
train = modelNmake(train)

#주행거리 최댓값 삭제
max_mileage = train['주행거리'].max()
train = train[train['주행거리'] != max_mileage]


In [47]:

scaler = StandardScaler()

# 주행거리 열 정규화
train['주행거리'] = scaler.fit_transform(train[['주행거리']])

# 배기량 열 정규화
train['배기량'] = scaler.fit_transform(train[['배기량']])


In [48]:
#라벨인코딩
ordinal_features = ['브랜드', '차량모델명', '판매도시', '판매구역']

for feature in ordinal_features:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])
    
    for label in np.unique(test[feature]):
        if label not in le.classes_:
            le.classes_ = np.append(le.classes_, label)
    test[feature] = le.transform(test[feature])

In [49]:
train.head()

,ID,생산년도,모델출시년도,브랜드,차량모델명,판매도시,판매구역,주행거리,배기량,압축천연가스(CNG),경유,가솔린,하이브리드,액화석유가스(LPG),가격
0,TRAIN_00000,2018,2014,16,47,1214,12,-0.999720,-1.446895,0,0,1,0,0,51.74
1,TRAIN_00001,2010,2006,17,20,2137,13,-0.397240,-0.348665,0,0,1,0,0,41.47
2,TRAIN_00002,2002,2002,9,36,785,15,1.058122,0.014356,0,0,1,0,0,17.81
3,TRAIN_00003,2006,2001,11,133,546,15,0.849629,0.725730,0,1,0,0,0,18.20
4,TRAIN_00004,2007,2007,3,25,1839,5,1.007000,0.223368,0,1,0,0,0,17.55


In [50]:
#ID와 가격 드롭
print(train.columns)
train_x = train.drop(['ID', '가격'], axis = 1)
train_y = train['가격']

test_x = test.drop('ID', axis = 1)
print(train_x.columns)
print(train_x.info())

#
train_y = train['가격']

Index(['ID', '생산년도', '모델출시년도', '브랜드', '차량모델명', '판매도시', '판매구역', '주행거리', '배기량',
       '압축천연가스(CNG)', '경유', '가솔린', '하이브리드', '액화석유가스(LPG)', '가격'],
      dtype='object')
Index(['생산년도', '모델출시년도', '브랜드', '차량모델명', '판매도시', '판매구역', '주행거리', '배기량',
       '압축천연가스(CNG)', '경유', '가솔린', '하이브리드', '액화석유가스(LPG)'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
Int64Index: 57912 entries, 0 to 57919
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   생산년도         57912 non-null  int64  
 1   모델출시년도       57912 non-null  int64  
 2   브랜드          57912 non-null  int64  
 3   차량모델명        57912 non-null  int64  
 4   판매도시         57912 non-null  int64  
 5   판매구역         57912 non-null  int64  
 6   주행거리         57912 non-null  float64
 7   배기량          57912 non-null  float64
 8   압축천연가스(CNG)  57912 non-null  int64  
 9   경유           57912 non-null  int64  
 10  가솔린          57912 non-null  int64  
 11  하이브리드        57912 n

In [51]:
# train 데이터를 train과 validation으로 8:2로 분할


# train_x와 train_y를 훈련 데이터와 검증 데이터로 나누기
train_x, validation_x, train_y, validation_y = train_test_split(train_x, train_y, test_size=0.2, random_state=42)

print(train_x.shape, train_y.shape)
print(validation_x.shape, validation_y.shape)

(46329, 13) (46329,)
(11583, 13) (11583,)


In [52]:
#회귀모델 정의, 모델학습
dtrain = xgb.DMatrix(train_x, label=train_y)
dval = xgb.DMatrix(validation_x, label=validation_y)

# XGBoost 모델 하이퍼파라미터 설정
params = {
    'objective': 'reg:squarederror',  # 회귀 문제일 경우
    'eval_metric': 'mae',  # MAE 사용
    'eta': 0.1,  # 학습률
    'max_depth': 6,  # 트리 깊이
    'subsample': 0.8,  # 데이터 샘플 비율
    'colsample_bytree': 0.8  # 트리에서 사용하는 열 비율
}

num_round = 100  # 훈련 반복 횟수

# 모델 훈련
model = xgb.train(params, dtrain, num_round, evals=[(dval, 'eval')], early_stopping_rounds=10)

# 예측
predictions = model.predict(dval)

# 평가: MAE 계산
mae = mean_absolute_error(validation_y, predictions)
print(f'Mean Absolute Error (MAE): {mae}')

[0]	eval-mae:26.52867
[1]	eval-mae:24.32747
[2]	eval-mae:22.48549
[3]	eval-mae:20.87953
[4]	eval-mae:19.29142
[5]	eval-mae:17.88187
[6]	eval-mae:16.72105
[7]	eval-mae:15.70162
[8]	eval-mae:14.72318
[9]	eval-mae:13.86498
[10]	eval-mae:13.10946
[11]	eval-mae:12.45211
[12]	eval-mae:11.88223
[13]	eval-mae:11.39361
[14]	eval-mae:10.93893
[15]	eval-mae:10.55422
[16]	eval-mae:10.20855
[17]	eval-mae:9.92816
[18]	eval-mae:9.67920
[19]	eval-mae:9.43806
[20]	eval-mae:9.22057
[21]	eval-mae:9.04350
[22]	eval-mae:8.89015
[23]	eval-mae:8.73860
[24]	eval-mae:8.59878
[25]	eval-mae:8.48107
[26]	eval-mae:8.37557
[27]	eval-mae:8.28241
[28]	eval-mae:8.20077
[29]	eval-mae:8.13321
[30]	eval-mae:8.05273
[31]	eval-mae:7.97675
[32]	eval-mae:7.91175
[33]	eval-mae:7.87315
[34]	eval-mae:7.78840
[35]	eval-mae:7.74345
[36]	eval-mae:7.69033
[37]	eval-mae:7.63841
[38]	eval-mae:7.58417
[39]	eval-mae:7.55286
[40]	eval-mae:7.51546
[41]	eval-mae:7.48922
[42]	eval-mae:7.45433
[43]	eval-mae:7.42141
[44]	eval-mae:7.38158
[45

In [83]:
model_L = LGBMRegressor(max_depth=5, learning_rate=0.1, n_estimators=3000, random_state=42)
model_L.fit(train_x, train_y)

# 예측 및 평가
y_pred_L = model.predict(validation_x)  # validation_x를 사용하여 예측
mae = mean_absolute_error(validation_y, y_pred_L)
print(f'Mean Absolute Error (MAE): {mae:.2f}')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000911 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 973
[LightGBM] [Info] Number of data points in the train set: 46329, number of used features: 13
[LightGBM] [Info] Start training from score 52.225137
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [86]:
y_pred_L= model_L.predict(test_x)
print(y_pred_L)

submission = pd.read_csv('  자동차가격예측대회/sample_submission.csv')
submission['가격'] = y_pred_L
submission.to_csv('./submit_L.csv', index = False)
print("제출 파일이 생성되었습니다: submit.csv")

[ 81.53090033  88.71183186 116.29955934 ... 113.30744135 104.73924542
 116.45458469]
제출 파일이 생성되었습니다: submit.csv


In [ ]:
model = DecisionTreeRegressor(random_state = 42)
model.fit(train_x, train_y)
y_pred = model.predict(validation_x)
# MSE 계산
mae = mean_absolute_error(validation_y, y_pred)
print(f"Mean Squared Error (MSE): {mae}")

Mean Squared Error (MSE): 182.5770126430836


In [59]:
model = CatBoostRegressor(iterations=4000,  # 트리의 수
                          learning_rate=0.01,  # 학습률
                          depth=10,              # 트리 깊이
                          loss_function='MAE',  # 손실 함수 (Mean Absolute Error)
                          cat_features=[]      
                          )

# 모델 학습
model.fit(train_x, train_y, 
          eval_set=(validation_x, validation_y),  # 검증 데이터
          early_stopping_rounds=10,                # 10번의 라운드 동안 성능 향상이 없다면 학습 종료
          verbose=100)

# 예측
y_pred = model.predict(validation_x)

# 평가 (MAE)
mae = mean_absolute_error(validation_y, y_pred)
print(f'Mean Absolute Error: {mae}')

0:	learn: 27.7833603	test: 27.5798853	best: 27.5798853 (0)	total: 27.5ms	remaining: 1m 50s
100:	learn: 14.0176543	test: 14.0401835	best: 14.0401835 (100)	total: 1.32s	remaining: 51s
200:	learn: 9.8804453	test: 10.0011567	best: 10.0011567 (200)	total: 2.53s	remaining: 47.7s
300:	learn: 8.3751721	test: 8.5453264	best: 8.5453264 (300)	total: 3.79s	remaining: 46.6s
400:	learn: 7.6965142	test: 7.8819581	best: 7.8819581 (400)	total: 5.21s	remaining: 46.8s
500:	learn: 7.3141738	test: 7.5130276	best: 7.5130276 (500)	total: 6.43s	remaining: 44.9s
600:	learn: 7.0553402	test: 7.2696386	best: 7.2696386 (600)	total: 7.68s	remaining: 43.4s
700:	learn: 6.8413848	test: 7.0732297	best: 7.0732297 (700)	total: 8.89s	remaining: 41.8s
800:	learn: 6.6712873	test: 6.9246578	best: 6.9246578 (800)	total: 10.1s	remaining: 40.2s
900:	learn: 6.5284810	test: 6.8026783	best: 6.8026783 (900)	total: 11.2s	remaining: 38.6s
1000:	learn: 6.3900420	test: 6.6871617	best: 6.6871617 (1000)	total: 12.4s	remaining: 37s
1100:	

In [78]:
catboost_model = CatBoostRegressor(iterations=3000, learning_rate=0.05, depth=9, verbose=0)
xgboost_model = XGBRegressor(n_estimators=3000, learning_rate=0.05, max_depth=9)


ensemble_model = VotingRegressor(estimators=[('catboost', catboost_model), ('xgboost', xgboost_model)])


ensemble_model.fit(train_x, train_y)


y_pred_train = ensemble_model.predict(validation_x)

mae = mean_absolute_error(validation_y, y_pred_train)
print(f'Mean Absolute Error: {mae}')

Mean Absolute Error: 5.758783500346349


In [79]:
#test
y_pred_test = ensemble_model.predict(test_x)

In [ ]:
print(y_pred_test)
print()

[ 77.5857915   90.39117212  89.45502295 ...  93.32265661  82.9098292
 102.00557714]


In [82]:
submission = pd.read_csv('  자동차가격예측대회/sample_submission.csv')
submission['가격'] = y_pred_test
submission.to_csv('./submit.csv', index = False)
print("제출 파일이 생성되었습니다: submit.csv")

제출 파일이 생성되었습니다: submit.csv
